# Dependency report: what depends on what, beyond the code graph

**Goal.** Show every dependency between the artifacts of a codebase: the usual ones the compiler sees (the code graph from `jdeps` and the template edges) **and** the ones the
`dependency-map` skill adds (XML, strings, files, keys, URLs, tables, history). Explain for each kind **how it was searched** and **what breaks when its target changes**, and
illustrate what depends on what, so that the answer to "what falls over if I change this?" can be read directly.

**Sources** (only read, never changed):
- the edge list `dependencies.csv` written by `scripts/dependencies.py`,
- the coupled pairs `change-coupling.csv` written by `scripts/change_coupling.py` (optional),
- the code graph as an edge list `source,target,kind` written by a language tool or any script (the usual dependencies; optional, but without it the report shows only the unusual ones),
- the Git repository the edges refer to (to check that the edge list still matches it).

**Outputs** go to `output/` next to this notebook: figures as PNG and the summary tables as CSV.

In [ ]:
# ---------------------------------------------------------------- inputs (only read)
EDGES_CSV = "../../temp/dependency-map/dependencies.csv"           # edge list of dependencies.py
COUPLING_CSV = "../../temp/dependency-map/change-coupling.csv"     # pairs of change_coupling.py; None skips the history part
CODE_EDGES_CSV = "../../temp/dependency-map/code-edges.csv"        # the usual dependencies (class and template edges, source,target,kind; see SKILL.md); None shows only the unusual ones
REPO_ROOT = "../.."                                                # the repository the edges refer to

# ---------------------------------------------------------------- outputs
OUTPUT_DIR = "output"                                              # figures and tables, next to the notebook
SAVE_FIGURES = True                                                # also write every figure as PNG

# ---------------------------------------------------------------- what to look at
FOCUS_FILES = []                # repo-relative files for the "what breaks if I change this" figure; [] = the file most depended on in ways the compiler does not check
TOP_N = 8                       # rows in "most depended on" tables
SAMPLE_SIZE = 8                 # edges drawn at random per kind, for the spot check by a person
SAMPLE_SEED = 7                 # fixed seed: the same sample every run
EXAMPLES_PER_MODE = 3           # example files named in the focus figure
HIDDEN_ROWS = 15                # hidden pairs listed in the history section
EXAMPLES_PER_KIND = 2           # edges shown as code examples per kind
EXAMPLE_SEED = 11               # fixed seed for the examples
CONTEXT_LINES = 1               # lines shown above and below the line that makes a dependency
SNIPPET_WIDTH = 150             # characters of a line shown in an example
FENCE_LANG = {".py": "python", ".php": "php", ".rb": "ruby", ".c": "c", ".h": "c", ".ts": "typescript", ".toml": "toml", ".erb": "erb", ".hbs": "handlebars",
              ".ejs": "html", ".j2": "jinja", ".md": "markdown", ".java": "java", ".jsp": "html", ".tag": "html", ".html": "html", ".js": "javascript", ".xml": "xml", ".properties": "properties",
              ".sql": "sql", ".yml": "yaml", ".yaml": "yaml", ".json": "json"}   # syntax highlighting of the examples by file extension
MAX_ARROWS = 7                  # arrows per panel in the layer diagram: only the strongest flows are drawn

# ---------------------------------------------------------------- vocabulary of the report
KIND_ORDER = ["code-graph", "template-edge", "xml-class", "bean-ref", "orm-relation", "url-link", "data-table", "table-defined-in", "string-class", "file-name",
              "key-link", "bean-lookup", "prefix-class", "dynamic-site", "name-link", "co-change"]
TIER = {"code-graph": "code", "template-edge": "code", "xml-class": "use", "bean-ref": "use", "orm-relation": "use", "string-class": "use", "bean-lookup": "use", "prefix-class": "use", "file-name": "use",
        "url-link": "contract", "key-link": "contract", "name-link": "names", "data-table": "data", "table-defined-in": "data"}   # how a kind carries a change
FAILURE_MODE = {"code-graph": "compiler", "template-edge": "first request", "xml-class": "startup", "bean-ref": "startup", "orm-relation": "startup",
                "string-class": "runtime", "file-name": "runtime", "bean-lookup": "runtime", "prefix-class": "runtime",
                "url-link": "contract", "key-link": "contract", "name-link": "name"}                                              # how a dependent fails when the target is renamed
MODE_ORDER = ["compiler", "first request", "startup", "runtime", "contract", "name", "schema"]                                  # from caught early to caught late
MODE_CAUGHT_BY = {"compiler": "the compiler", "first request": "the first request of that page", "startup": "starting the application",
                  "runtime": "only a test on that path", "contract": "only a test or a link check", "name": "usually nothing: it fails or shows nothing",
                  "schema": "a test against the database"}
EARLY_MODES = ["compiler", "startup"]                              # caught before the code runs for a user
LATE_MODES = ["runtime", "contract", "name"]                       # the compiler does not check these
DIAGRAM_KINDS = ["code-graph", "template-edge", "xml-class", "bean-ref", "orm-relation", "string-class", "bean-lookup", "prefix-class", "file-name",
                 "url-link", "key-link", "data-table", "table-defined-in"]            # name-link is left out of the layer diagram: the weakest evidence and the most numerous

# ---------------------------------------------------------------- layers and artifact types (first match wins)
LAYERS = []                     # [] = by folder (see SOURCE_ROOTS); or [("name", "regex on the path"), ...], first match wins, e.g. [("ui", r"\.jsp$"), ("data access", r"/dao/")]
SOURCE_ROOTS = ["src", "lib", "app", "apps", "packages", "source"]   # a folder below one of these is its own layer (src/flask, lib/sinatra); otherwise the first folder is the layer
LAYER_ORDER = []                # [] = the layer with the most edges first; or a list of layer names, around the ellipse of the layer diagram (neighbours are drawn next to each other)
EXT_GROUP = {"py": "Python", "php": "PHP", "rb": "Ruby", "rake": "Ruby", "c": "C/C++", "h": "C/C++", "cc": "C/C++", "cpp": "C/C++", "hpp": "C/C++", "ts": "TypeScript",
             "tsx": "TypeScript", "mjs": "JavaScript", "cjs": "JavaScript", "jsx": "JavaScript", "toml": "TOML", "ini": "INI", "cfg": "INI", "erb": "JSP / template",
             "hbs": "JSP / template", "ejs": "JSP / template", "twig": "JSP / template", "j2": "JSP / template", "jinja": "JSP / template", "phtml": "JSP / template",
             "md": "docs", "rst": "docs", "txt": "text",
             "java": "Java", "jsp": "JSP / template", "tag": "JSP / template", "html": "HTML", "js": "JavaScript", "xml": "XML", "properties": "properties",
             "json": "JSON", "yml": "YAML", "yaml": "YAML", "sql": "SQL", "csv": "CSV"}

# ---------------------------------------------------------------- colours (validated palette, slots 1-3, plus neutrals)
COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "gray": "#a8a7a0", "ink": "#0b0b0b", "ink2": "#52514e", "surface": "#fcfcfb"}
TIER_COLOR = {"code": "ink2", "use": "blue", "contract": "orange", "data": "aqua"}

## Dependencies

The notebook needs `pandas`, `matplotlib`, `numpy` and `ipython` (for the examples) and `git` to check the edge list against the repository.

In [ ]:
%%bash
pip install --quiet pandas matplotlib numpy ipython
command -v git > /dev/null || { echo "git is required"; exit 1; }

## Imports

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## Load the edge list

One row per dependency: `source` depends on `target`. Every row names its **kind**, the **layer** (`structure`: a change in the target
can break the source; `evidence`: only a hint), the **confidence** and the **evidence** (file and line). The file is read as it is;
nothing is changed in it.

In [ ]:
edges = pd.read_csv(EDGES_CSV, dtype={"weight": "int64"})
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(exist_ok=True)
print(f"{len(edges):,} edges read from {EDGES_CSV}; the first five:")
edges.head()

### Check: is the edge list usable?

The columns must be the ones this notebook relies on, every kind must be one the notebook explains, and no key column may be empty.
A kind that the notebook does not know would silently be left out of the report, so the check fails instead.

In [ ]:
expected = {"source", "target", "kind", "layer", "confidence", "evidence", "weight"}
assert expected <= set(edges.columns), f"missing columns: {expected - set(edges.columns)}"
assert len(edges) > 0, "the edge list is empty"
assert not edges[["source", "target", "kind"]].isna().any().any(), edges[edges[["source", "target", "kind"]].isna().any(axis=1)].head()
unknown = sorted(set(edges["kind"]) - set(KIND_ORDER))
assert not unknown, f"kinds the report does not explain: {unknown}"
print(f"check passed: all columns present, no empty key, all {edges['kind'].nunique()} kinds are explained in this notebook")

## The usual dependencies: the code graph

The dependencies a language tool sees come from a second file, the code graph edge list that such a tool or script writes (see `SKILL.md`): class to class, for example from `jdeps`, and the template
edges between servlets, JSPs and classes. They are read the same way and put in one table with the others, as the kinds `code-graph` and `template-edge`, so that
the usual and the unusual dependencies can be compared in the same tables and figures. Without that file (`CODE_EDGES_CSV = None`) only the unusual ones are shown.

In [ ]:
code_edges = pd.read_csv(CODE_EDGES_CSV) if CODE_EDGES_CSV else pd.DataFrame(columns=["source", "target", "kind"])
code_graph = code_edges.assign(kind=lambda d: np.where(d["kind"] == "template", "template-edge", "code-graph"), layer="structure",
                               confidence="verified by bytecode and template scan", evidence="code graph", weight=1)
all_edges = pd.concat([edges, code_graph], ignore_index=True)
assert not code_edges.empty or CODE_EDGES_CSV is None, "the code graph file is empty"
print(f"{len(code_graph):,} code-graph and template edges read; {len(all_edges):,} edges in all. The first five of the code graph:")
code_graph.head()

### Check: does the edge list still match the repository?

An edge list written for another commit points at files that are gone or renamed. Every endpoint that is a file (not a table node or a run-time lookup node) has to
be a tracked file of the repository, for the code graph as well as for the other edges; otherwise the scripts must run again.

In [ ]:
tracked = set(subprocess.run(["git", "-C", REPO_ROOT, "ls-files"], capture_output=True, text=True, check=True).stdout.splitlines())
endpoints = pd.concat([all_edges["source"], all_edges["target"]]).drop_duplicates()
file_endpoints = endpoints[~endpoints.str.startswith(("table:", "dynamic:"))]
stale = file_endpoints[~file_endpoints.isin(tracked)]
assert stale.empty, f"{len(stale)} endpoints are not tracked files (edge list from another commit?):\n{stale.head()}"
print(f"check passed: all {len(file_endpoints):,} file endpoints are tracked files of the repository")

## Overview: how many dependencies of which kind

The table counts, per kind, the edges and the distinct files on either end, and shows the confidence the kind carries. The bar chart shows
the edge counts, coloured by how the kind carries a change: a **code** edge is what the compiler checks, a **use** edge carries a change like a call does, a **contract** edge names a URL or a
key, a **data** edge goes through a database table, and **names** and **evidence** are weaker hints.

In [ ]:
per_kind = (all_edges.groupby("kind")
                 .agg(edges=("source", "size"), sources=("source", "nunique"), targets=("target", "nunique"), confidence=("confidence", "first"))
                 .reindex(KIND_ORDER).dropna(subset=["edges"]).astype({"edges": int, "sources": int, "targets": int}))
assert per_kind["edges"].sum() == len(all_edges), "an edge was lost in the aggregation"
per_kind

**Figure: edges per kind.** One measure, so one axis. The bars are labelled directly with their counts, and the legend names the three tiers. The table
above carries the same numbers.

In [ ]:
bar_color = per_kind.index.map(lambda k: TIER_COLOR.get(TIER.get(k), "gray")).map(COLORS)
fig, ax = plt.subplots(figsize=(9, 0.42 * len(per_kind) + 1.2), facecolor=COLORS["surface"])
ax.set_facecolor(COLORS["surface"])
bars = ax.barh(per_kind.index[::-1], per_kind["edges"][::-1], color=bar_color[::-1], height=0.62)
ax.bar_label(bars, padding=4, color=COLORS["ink2"], fontsize=9)
ax.set_xlabel("Edges (dependencies found)", color=COLORS["ink2"])
ax.set_title("Dependencies found, by kind", loc="left", color=COLORS["ink"], fontsize=12)
ax.tick_params(colors=COLORS["ink2"], labelsize=9)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
legend_items = [("code: the compiler sees it", "ink2"), ("use: carries a change like a call", "blue"), ("contract: a URL or a key", "orange"), ("data: a shared table", "aqua"), ("names, run-time sites, history: hints", "gray")]
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLORS[c]) for _, c in legend_items], labels=[t for t, _ in legend_items],
          frameon=False, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=2, fontsize=9, labelcolor=COLORS["ink2"])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_PATH / "edges-per-kind.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
plt.show()

## Illustration 1: which kind of artifact depends on which

Each file gets an artifact type from its extension (a table node and a run-time lookup node have their own types). The matrix counts the
structure edges between the types, for the kinds in `DIAGRAM_KINDS` (the name links are left out: the weakest evidence and the most numerous, they would drown
the rest): **rows depend on columns**. A dark cell off the diagonal is a dependency that crosses a technology border, which is
exactly where the compiler cannot help.

In [ ]:
ext = lambda s: s.str.extract(r"\.([A-Za-z0-9]+)$", expand=False).map(EXT_GROUP)
structure = all_edges.query("layer == 'structure' and kind in @DIAGRAM_KINDS").assign(
    source_type=lambda d: ext(d["source"]).fillna("other").mask(d["source"].str.startswith("table:"), "table").mask(d["source"].str.startswith("dynamic:"), "run-time lookup"),
    target_type=lambda d: ext(d["target"]).fillna("other").mask(d["target"].str.startswith("table:"), "table").mask(d["target"].str.startswith("dynamic:"), "run-time lookup"))
type_matrix = pd.crosstab(structure["source_type"], structure["target_type"])
assert type_matrix.to_numpy().sum() == len(structure), "the matrix lost edges"
type_matrix

**Figure: dependencies between artifact types.** The colour is one hue, light to dark, for the number of edges; every cell with edges also carries its number, so
the figure can be read without the colour. The same numbers are in the table above.

In [ ]:
ramp = LinearSegmentedColormap.from_list("depth", [COLORS["surface"], COLORS["blue"], "#0d2f5c"])
order_rows = type_matrix.sum(axis=1).sort_values(ascending=False).index
order_cols = type_matrix.sum(axis=0).sort_values(ascending=False).index
matrix = type_matrix.loc[order_rows, order_cols]
fig, ax = plt.subplots(figsize=(1.0 * len(order_cols) + 3, 0.6 * len(order_rows) + 2), facecolor=COLORS["surface"])
ax.imshow(np.log1p(matrix.to_numpy()), cmap=ramp, aspect="auto")
ax.set_xticks(range(len(order_cols)), order_cols, rotation=35, ha="right")
ax.set_yticks(range(len(order_rows)), order_rows)
ax.set_xlabel("depends on (target type)", color=COLORS["ink2"])
ax.set_ylabel("source type", color=COLORS["ink2"])
ax.set_title("Which artifact depends on which (edges, colour on a log scale)", loc="left", color=COLORS["ink"], fontsize=12)
ax.tick_params(colors=COLORS["ink2"], labelsize=9)
for (r, c), n in np.ndenumerate(matrix.to_numpy()):
    if n:
        ax.text(c, r, f"{n}", ha="center", va="center", fontsize=8, color="white" if np.log1p(n) > 0.55 * np.log1p(matrix.to_numpy().max()) else COLORS["ink"])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_PATH / "artifact-type-matrix.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
plt.show()

## Showing the dependencies as they look in the code

Each kind below ends with a few edges shown the way they appear in the files: the **line in the source** that creates the dependency, and the **line in the target**
that it depends on. The examples are read from the repository when the notebook runs, so they always match the edge list. Three small helpers do this; they are
needed in every section and cannot be a pandas operation: one finds the line of a pattern in a file, one prints a few numbered lines around a line (the hit marked
with `>`), and one displays a source and its target together. An edge that comes from a scan of a whole file has no single line and says so.

In [ ]:
def find_line(path, pattern, flags=0):
    text = Path(REPO_ROOT, path).read_text(errors="replace")
    if path.endswith(".xml"):               # a commented-out element is not a definition
        text = re.sub(r"<!--.*?-->", lambda m: re.sub(r"[^\n]", " ", m.group(0)), text, flags=re.S)
    text = text.splitlines()
    return next((i + 1 for i, line in enumerate(text) if re.search(pattern, line, flags)), 0)


def snippet(path, line):
    if not line:
        return "(no single line: the edge comes from a scan of the whole file)"
    text = Path(REPO_ROOT, path).read_text(errors="replace").splitlines()
    lo, hi = max(line - 1 - CONTEXT_LINES, 0), min(line + CONTEXT_LINES, len(text))
    return "\n".join(f"{'>' if i + 1 == line else ' '}{i + 1:>5}  {text[i][:SNIPPET_WIDTH].rstrip()}" for i in range(lo, hi))


def show_example(kind, source, source_line, target=None, target_line=0, note=""):
    fence = lambda path, line: f"```{FENCE_LANG.get(Path(path).suffix, 'text')}\n{snippet(path, line)}\n```"
    parts = [f"**{kind}** {note}".strip(), f"`{source}`", fence(source, source_line)]
    if target:
        parts += ["depends on", f"`{target}`", fence(target, target_line)]
    display(Markdown("\n\n".join(parts)))

## A class uses another class (the usual dependency) (`code-graph`)

**How it is found.** The compiled classes are read with `jdeps -verbose:class -filter:none`; without the last option `jdeps` silently drops every dependency inside one package, which are the closest couplings there are. Each class is mapped back to its source file through its package and file name. An edge says: this file's compiled class refers to that file's class.

**The dependency problem.** The compiler enforces this edge. Change a signature, rename or delete the class and the build fails, which is the best case. What the compiler does not say is what changes in behaviour: for that, the callers, followed transitively, are the radius.

**What catches it.** The compiler.

**Where it is weak.** Only what is compiled and only static references. Reflection, strings and configuration are invisible to it, which is why the other kinds exist.

In [ ]:
kind_code_graph = all_edges.query("kind == 'code-graph'")
kind_code_graph.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `code-graph` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_code_graph.sample(min(len(kind_code_graph), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `code-graph` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_code_graph.empty:
    print("no edges of this kind")

for r in kind_code_graph.sample(min(len(kind_code_graph), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    stem = Path(r.target).stem
    line = find_line(r.source, rf"\bimport\s+[\w.]*\.{stem}\s*;") or find_line(r.source, rf"\b{stem}\b")
    show_example("code-graph", r.source, line, r.target, find_line(r.target, rf"\b(class|interface|enum)\s+{stem}\b"))

## A template and its classes (`template-edge`)

**How it is found.** These edges are not found by this skill; they come with the code graph edge list, written by whatever scanned the views. For a Java web application that is a scan of what links views and code: a servlet that forwards to a page (the `Page` enum names the JSP), a JSP's `import` and `useBean` directives, and `include` between templates. Each becomes an edge between the class and the template or between two templates.

**The dependency problem.** A template depends on the class it imports and on the pages it includes, and a servlet depends on the page it forwards to. A JSP is compiled at the first request of the page, so a break shows late, not at build time.

**What catches it.** The first request of that page.

**Where it is weak.** Includes with a computed path are missed.

In [ ]:
kind_template_edge = all_edges.query("kind == 'template-edge'")
kind_template_edge.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `template-edge` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_template_edge.sample(min(len(kind_template_edge), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `template-edge` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_template_edge.empty:
    print("no edges of this kind")

for r in kind_template_edge.sample(min(len(kind_template_edge), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    is_page = r.target.endswith((".jsp", ".tag"))
    name = Path(r.target).name if is_page else Path(r.target).stem
    show_example("template-edge", r.source, find_line(r.source, re.escape(name)), r.target, 1 if is_page else find_line(r.target, rf"\b(class|interface|enum)\s+{name}\b"))

## XML names a class (`xml-class`)

**How it is found.** Every XML file is searched for attributes and elements that carry a Java class name: a bean's `class="…"`, a `web.xml` `<servlet-class>`, `<filter-class>` or `<listener-class>`, and a Hibernate `<class name="…">`. The full class name is resolved to the file that declares it, through the package line and the file name. The match is exact, so the edge is verified by syntax.

**The dependency problem.** The XML file depends on the class by its full name. Rename, move or delete the class and nothing in the code graph complains, because XML is not code: the application fails when the context loads, at start-up.

**What catches it.** Starting the application (a smoke test).

**Where it is weak.** Only names written in XML. Wiring done by annotation scanning is not declared anywhere and is not found.

In [ ]:
kind_xml_class = all_edges.query("kind == 'xml-class'")
kind_xml_class.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `xml-class` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_xml_class.sample(min(len(kind_xml_class), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `xml-class` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_xml_class.empty:
    print("no edges of this kind")

for r in kind_xml_class.sample(min(len(kind_xml_class), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    stem = Path(r.target).stem
    show_example("xml-class", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), r.target, find_line(r.target, rf"\b(class|interface|enum)\s+{stem}\b"))

## A bean refers to another bean (`bean-ref`)

**How it is found.** Spring bean files are parsed as XML trees. Inside a bean, `ref`, `bean` and `local` attributes point at the id of another bean; both beans are resolved to their classes, and the edge goes from the class of the referring bean to the class of the referenced one.

**The dependency problem.** The class of a bean depends on another class through an id the compiler never reads. Rename the id or remove the bean and the wiring breaks at start-up, far from the code that changed.

**What catches it.** Starting the application.

**Where it is weak.** Only XML wiring. References by type (`@Autowired`) are not declared in XML.

In [ ]:
kind_bean_ref = all_edges.query("kind == 'bean-ref'")
kind_bean_ref.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `bean-ref` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_bean_ref.sample(min(len(kind_bean_ref), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `bean-ref` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_bean_ref.empty:
    print("no edges of this kind")

for r in kind_bean_ref.sample(min(len(kind_bean_ref), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    bean = re.search(r"`([^`]+)`", r.evidence).group(1)
    xml = r.evidence.rsplit(" in ", 1)[-1]
    show_example("bean-ref", xml, find_line(xml, rf'(ref|bean|local)\s*=\s*"{bean}"'), xml, find_line(xml, rf'\b(id|name)\s*=\s*"{bean}"'),
                 note=f"bean `{bean}`: the class {Path(r.source).stem} uses {Path(r.target).stem}")

## A mapped class relates to another (`orm-relation`)

**How it is found.** Hibernate mapping files are parsed as trees; relations (`class`, `entity-name`) inside a mapped class give an edge from the mapped class to the related class.

**The dependency problem.** A mapped class depends on the class of its relation. With annotations the language tool already sees this; this kind exists for XML mappings, and an empty result means the mappings are annotations.

**What catches it.** Starting the application (the session factory fails to build).

**Where it is weak.** Only XML mappings.

In [ ]:
kind_orm_relation = all_edges.query("kind == 'orm-relation'")
kind_orm_relation.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `orm-relation` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_orm_relation.sample(min(len(kind_orm_relation), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

## A page or class names a servlet's URL (`url-link`)

**How it is found.** The servlet mappings of `web.xml` give, for each servlet class, its URL patterns. Templates, scripts and classes are then searched for those URLs: a name in quotes or after a slash, at least a few characters long, and not in a vendored library. The match is by text, so the edge is a heuristic.

**The dependency problem.** The source depends on the **contract** of the servlet: its URL and parameters, not its inside. Change the URL or the mapping and the link answers 404 or leads to another page; the servlet's inside can change freely. No build step notices.

**What catches it.** Only a test that follows the link, or a link check.

**Where it is weak.** A URL pattern that is an ordinary word matches ordinary text. Vendored libraries are skipped by name.

In [ ]:
kind_url_link = all_edges.query("kind == 'url-link'")
kind_url_link.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `url-link` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_url_link.sample(min(len(kind_url_link), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `url-link` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_url_link.empty:
    print("no edges of this kind")

for r in kind_url_link.sample(min(len(kind_url_link), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    pattern = re.search(r"`([^`]+)`", r.evidence).group(1)
    tgt = r.target
    web_xml = next(iter(all_edges.query("kind == 'xml-class' and target == @tgt and source.str.endswith('web.xml')")["source"]), None)
    show_example("url-link", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), web_xml,
                 find_line(web_xml, rf"<url-pattern>\s*/?{pattern}\b") if web_xml else 0, note=f"URL `{pattern}`")

## A class, mapping or template uses a database table (`data-table`)

**How it is found.** Tables are found first, in Liquibase changelogs (`createTable`, `tableName=`) and SQL scripts (`create table`). Then classes, mappings, templates and XML query files are searched for an SQL keyword (`from`, `join`, `into`, `update`, `table`, …) followed by a known table name, and for `@Table` and `table=` annotations. Each hit is an edge to a node `table:<name>`; the table node itself depends on the changelog or script that defines it (`table-defined-in`).

**The dependency problem.** Everyone who names a table depends on its name and columns. A schema change in a changelog breaks all of them at run time, and two classes that use the same table are coupled through the data even if they never call each other.

**What catches it.** Only a test against the database.

**Where it is weak.** A table with a common name (`customer`, `status`) matches widely. This is shared data, not a call, so the radius follows these edges only one step.

In [ ]:
kind_data_table = all_edges.query("kind == 'data-table'")
kind_data_table.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `data-table` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_data_table.sample(min(len(kind_data_table), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `data-table` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_data_table.empty:
    print("no edges of this kind")

for r in kind_data_table.sample(min(len(kind_data_table), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    table, tgt = r.target.removeprefix("table:"), r.target
    defined = next(iter(all_edges.query("kind == 'table-defined-in' and source == @tgt")["target"]), None)
    show_example("data-table", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), defined,
                 find_line(defined, rf'tableName\s*=\s*"{table}"|create\s+table\s+(if\s+not\s+exists\s+)?{table}\b', re.I) if defined else 0, note=f"table `{table}`")

## A class is named by its full name as text (`string-class`)

**How it is found.** Every file is searched for the full name of a project class as text; in Java code only inside string literals, elsewhere anywhere. The match is exact, so the edge is verified by name. This finds JSP `import` directives, reflection by name, class names in JSON, properties and seed data.

**The dependency problem.** Reflection, templates and data name classes as text. Rename the class and the compiler stays silent; the failure comes when that text is read.

**What catches it.** Only a test that runs that code path.

**Where it is weak.** A class mentioned in a comment or a log text is a chance hit.

In [ ]:
kind_string_class = all_edges.query("kind == 'string-class'")
kind_string_class.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `string-class` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_string_class.sample(min(len(kind_string_class), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `string-class` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_string_class.empty:
    print("no edges of this kind")

for r in kind_string_class.sample(min(len(kind_string_class), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    stem = Path(r.target).stem
    show_example("string-class", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), r.target, find_line(r.target, rf"\b(class|interface|enum)\s+{stem}\b"))

## A string is the name of another file (`file-name`)

**How it is found.** Every string that equals the name of another tracked file, extension included and at least a minimum length, becomes an edge to that file. If the text names a directory too (`include/footer.jsp`), that directory part picks the file first; where the same name still exists in several places (`web` and `ws`), the nearest file by path wins. This finds includes, resource loading and the query files a data access class reads.

**The dependency problem.** A program that loads a resource, includes a template or reads a query file by name depends on that file. Rename or move it and the include or the load fails when it runs: a blank part of a page, a missing query.

**What catches it.** Only a test that runs that path.

**Where it is weak.** A name that exists in several places is resolved by proximity, which can pick the wrong one.

In [ ]:
kind_file_name = all_edges.query("kind == 'file-name'")
kind_file_name.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `file-name` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_file_name.sample(min(len(kind_file_name), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `file-name` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_file_name.empty:
    print("no edges of this kind")

for r in kind_file_name.sample(min(len(kind_file_name), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    show_example("file-name", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), r.target, 1)

## A rare key is spelled the same in two places (`key-link`)

**How it is found.** Rare, identifier-like strings are collected from Java string literals, from data files (XML text and attributes, properties keys, JSON) and from views (JSP, HTML, JavaScript words, vendored libraries skipped). A key is linked when it occurs in both places and is rare in each. Message keys, query names and values in seed data are found this way; for a view the edge points from the view to the code that sets the key.

**The dependency problem.** A message key, a query name or a request attribute is a name that must match in two places. Rename one side and a lookup fails, a text turns into its key, or a page renders empty, often without an error message.

**What catches it.** Only a test or a link check; often nothing.

**Where it is weak.** A key that merely appears in two places is not always a dependency (two files may show the same message). The edges are leads.

In [ ]:
kind_key_link = all_edges.query("kind == 'key-link'")
kind_key_link.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `key-link` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_key_link.sample(min(len(kind_key_link), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `key-link` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_key_link.empty:
    print("no edges of this kind")

for r in kind_key_link.sample(min(len(kind_key_link), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    key = re.search(r"`([^`]+)`", r.evidence).group(1)
    source_line, target_line = [int(x) for x in re.findall(r"line (\d+)", r.evidence)]
    show_example("key-link", r.source, source_line, r.target, target_line, note=f"key `{key}`")

## Code looks a bean up by its id (`bean-lookup`)

**How it is found.** Java code is searched for `getBean("id")`, `@Qualifier("id")` and `@Resource(name = "id")`. The id is resolved through the bean definitions to the bean's class, and the edge goes from the calling file to that class.

**The dependency problem.** The calling code depends on the class of the bean through its id. Rename the id or the bean and the lookup fails when that code runs.

**What catches it.** Only a test that runs that path.

**Where it is weak.** Only literal ids. A computed id is reported as a dynamic site.

In [ ]:
kind_bean_lookup = all_edges.query("kind == 'bean-lookup'")
kind_bean_lookup.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `bean-lookup` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_bean_lookup.sample(min(len(kind_bean_lookup), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `bean-lookup` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_bean_lookup.empty:
    print("no edges of this kind")

for r in kind_bean_lookup.sample(min(len(kind_bean_lookup), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    bean, tgt = re.search(r"`([^`]+)`", r.evidence).group(1), r.target
    xmls = all_edges.query("kind == 'xml-class' and target == @tgt")["source"].tolist()
    definition = rf'<bean\b[^>]*\bid\s*=\s*"{bean}"'
    xml = next((x for x in xmls if find_line(x, definition)), None)
    show_example("bean-lookup", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), xml,
                 find_line(xml, definition) if xml else 0, note=f"bean id `{bean}`")

## A class name is built from a package at run time (`prefix-class`)

**How it is found.** Where code calls `forName` or `loadClass` with a computed argument and the same file spells a package name as a string, the file is linked to every class of that package. The edge is a heuristic: the package is a strong hint for where the built name ends.

**The dependency problem.** Classes in that package are created by a name that is assembled at run time, here often from text a user typed. No caller in the code graph names them, so they look unused and get renamed or deleted. The failure comes only when that name is typed.

**What catches it.** Only a test that uses each name.

**Where it is weak.** Only when the package is spelled out in the same file.

In [ ]:
kind_prefix_class = all_edges.query("kind == 'prefix-class'")
kind_prefix_class.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `prefix-class` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_prefix_class.sample(min(len(kind_prefix_class), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `prefix-class` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_prefix_class.empty:
    print("no edges of this kind")

for r in kind_prefix_class.sample(min(len(kind_prefix_class), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    stem = Path(r.target).stem
    show_example("prefix-class", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), r.target, find_line(r.target, rf"\b(class|interface|enum)\s+{stem}\b"),
                 note="the package is spelled out in the same file as the lookup")

## Names are built or looked up at run time (`dynamic-site`)

**How it is found.** Calls to `forName`, `loadClass`, `getBean`, and reflective `getMethod` and `getField` are searched for an argument that is not a literal, not a constant and not a type. A call without an argument is a getter, and `getMethod` in a file that does not use reflection is another API.

**The dependency problem.** The target of the call is not known. Whatever the computed name resolves to can break, so a radius computed from the other edges is only a lower bound near these files. This kind is a warning flag, not an edge to a target.

**What catches it.** Nothing can say what it resolves to without running it.

**Where it is weak.** A few false alarms remain; each site is worth reading.

In [ ]:
kind_dynamic_site = all_edges.query("kind == 'dynamic-site'")
kind_dynamic_site.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `dynamic-site` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_dynamic_site.sample(min(len(kind_dynamic_site), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `dynamic-site` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_dynamic_site.empty:
    print("no edges of this kind")

for r in kind_dynamic_site.sample(min(len(kind_dynamic_site), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    show_example("dynamic-site", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), note="the target is only known when this line runs")

## An identifier is spelled the same elsewhere (`name-link`)

**How it is found.** Identifiers declared in a few Java classes (class names, properties behind getters, private fields, written in camelCase with several words) are searched as text in all other files; in Java code only as an exact string literal. Names found in too many files are reported as hubs and get no edges.

**The dependency problem.** Template expressions, bean properties and framework bindings need the same spelling as the declaration. Rename the property and a page renders empty or throws, with no compile error.

**What catches it.** Usually nothing.

**Where it is weak.** The weakest evidence: two things can share a name without depending on each other. By far the most numerous kind, so it is followed only one step.

In [ ]:
kind_name_link = all_edges.query("kind == 'name-link'")
kind_name_link.groupby("target").agg(dependents=("source", "nunique"), confidence=("confidence", "first")).nlargest(TOP_N, "dependents")

**Spot check.** A random sample of the `name-link` edges with their evidence, for a person to read against the source before relying on this kind. The
seed is fixed, so the sample is the same on every run.

In [ ]:
kind_name_link.sample(min(len(kind_name_link), SAMPLE_SIZE), random_state=SAMPLE_SEED)[["source", "target", "evidence"]]

**Example in the code.** A few `name-link` edges drawn at random, shown as they appear in the files: the line in the source that creates the dependency and, where the edge has one, the line in
the target that it depends on.

In [ ]:
if kind_name_link.empty:
    print("no edges of this kind")

for r in kind_name_link.sample(min(len(kind_name_link), EXAMPLES_PER_KIND), random_state=EXAMPLE_SEED).itertuples():
    name = re.search(r"`([^`]+)`", r.evidence).group(1)
    show_example("name-link", r.source, int(re.search(r"line (\d+)", r.evidence).group(1)), r.target, find_line(r.target, rf"\b{name}\b"), note=f"name `{name}`")

## Illustration 2: the layers and what depends on what

The files are sorted into layers by their path: by folder (the first folder, or the one below a source root such as `src` or `lib`) or, if `LAYERS` is set, by the first matching pattern; a table node and a run-time lookup node have their own. Every arrow reads **"depends on"**:
from the layer that has the dependency to the layer that is depended on. One panel per way a kind carries a change, so that the arrows of one kind can be followed
without the others in the way: **use** edges carry a change like a call does, **contract** edges name a URL or a key, **data** edges go through a database table.
Only the strongest flows of each panel are drawn (`MAX_ARROWS`) and only the layers that such an arrow touches are shown; the count is written on the arrow, and edges inside one layer are written in its box.

In [ ]:
if LAYERS:
    plain = [(name, re.sub(r"\((?!\?)", "(?:", rx)) for name, rx in LAYERS]   # no capture groups: contains() only matches
    names = [n for n, _ in LAYERS]
    layer_of = lambda s: pd.Series(np.select([s.str.contains(rx, regex=True) for _, rx in plain], names, default="other"), index=s.index)
else:
    parts = lambda s: s.str.split("/")
    layer_of = lambda s: parts(s).str[0].where(~(parts(s).str[0].isin(SOURCE_ROOTS) & (s.str.count("/") >= 2)), parts(s).str[:2].str.join("/")).mask(~s.str.contains("/"), "(root)")
layer_of_node = lambda s: layer_of(s).mask(s.str.startswith("table:"), "database table").mask(s.str.startswith("dynamic:"), "run-time lookup")
layered = all_edges.assign(source_layer=layer_of_node(all_edges["source"]), target_layer=layer_of_node(all_edges["target"]),
                       tier=all_edges["kind"].map(TIER)).query("layer == 'structure' and kind in @DIAGRAM_KINDS")
flows = (layered.groupby(["tier", "source_layer", "target_layer"]).size().rename("edges").reset_index())
assert flows["edges"].sum() == len(layered), "an edge was lost between the layers"
layer_order = LAYER_ORDER or pd.concat([layered["source_layer"], layered["target_layer"]]).value_counts().index.tolist()
assert set(flows["source_layer"]) | set(flows["target_layer"]) <= set(layer_order), "a layer is missing in LAYER_ORDER"
print(f"check passed: all {len(layered):,} structure edges are counted once in the flows; the strongest flows:")
flows.sort_values("edges", ascending=False).head(TOP_N * 2)

**Figure: layers and their dependencies, one panel per tier.** The title names the tier, the counts sit on the arrows
and in the boxes, so nothing depends on colour alone; the table above has the numbers behind all four panels. The layers sit on an ellipse, so that no arrow runs through a box; two arrows between
the same pair of layers bend to opposite sides and stay apart. The panels differ in which layers they show, so the layout is not the same in each.

In [ ]:
tiers = ["code", "use", "contract", "data"]
tier_title = {"code": "code: the compiler sees it", "use": "use: carries a change like a call", "contract": "contract: a URL or a key", "data": "data: a shared table"}
fig, axes = plt.subplots(2, 2, figsize=(15, 12), facecolor=COLORS["surface"])
for ax, tier in zip(axes.ravel(), tiers):
    ax.set_facecolor(COLORS["surface"])
    ax.set_xlim(-6.2, 6.2)
    ax.set_ylim(-4.4, 4.4)
    ax.axis("off")
    panel = flows.query("tier == @tier")
    strongest = panel.query("source_layer != target_layer").nlargest(MAX_ARROWS, "edges")
    shown = [layer for layer in layer_order if layer in set(strongest["source_layer"]) | set(strongest["target_layer"])]
    angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, max(len(shown), 1), endpoint=False)
    positions = {layer: (4.6 * np.cos(angle), 3.5 * np.sin(angle)) for layer, angle in zip(shown, angles)}
    inside = panel.query("source_layer == target_layer").groupby("source_layer")["edges"].sum()
    boxes = {}
    for layer, (x, y) in positions.items():
        boxes[layer] = FancyBboxPatch((x - 1.25, y - 0.3), 2.5, 0.6, boxstyle="round,pad=0.02,rounding_size=0.08", fc="white", ec=COLORS["gray"], lw=1)
        ax.add_patch(boxes[layer])
        ax.text(x, y + (0.09 if inside.get(layer, 0) else 0), layer, ha="center", va="center", fontsize=10, color=COLORS["ink"])
        if inside.get(layer, 0):
            ax.text(x, y - 0.14, f"{inside[layer]} inside", ha="center", va="center", fontsize=8, color=COLORS["ink2"])
    for i, row in enumerate(strongest.itertuples()):
        (x0, y0), (x1, y1) = positions[row.source_layer], positions[row.target_layer]
        t = 0.30 + 0.16 * (i % 3)                                   # staggered along the arrow so that neighbouring counts do not collide
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), patchA=boxes[row.source_layer], patchB=boxes[row.target_layer], connectionstyle="arc3,rad=0.14",
                                     arrowstyle="-|>", mutation_scale=12, lw=0.8 + 4.2 * np.sqrt(row.edges / strongest["edges"].max()),
                                     color=COLORS[TIER_COLOR[tier]], alpha=0.9))
        ax.text(x0 + t * (x1 - x0) + 0.07 * (y1 - y0), y0 + t * (y1 - y0) - 0.07 * (x1 - x0), f"{row.edges}", ha="center", va="center", fontsize=9,
                color=COLORS["ink"], bbox=dict(fc=COLORS["surface"], ec="none", pad=1.2))
    ax.set_title(tier_title[tier], loc="left", color=COLORS["ink"], fontsize=11)
fig.suptitle("Arrow: the layer it starts at depends on the layer it points to", x=0.01, y=0.97, ha="left", color=COLORS["ink"], fontsize=12)
fig.subplots_adjust(hspace=0.12, wspace=0.04, top=0.93)
if SAVE_FIGURES:
    fig.savefig(OUTPUT_PATH / "layer-dependencies.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
plt.show()

## Illustration 3: what falls over if I change this?

Pick the files that are about to change (`FOCUS_FILES`). Without a choice the notebook takes the file that the most other files depend on **in a way the
compiler does not check**. For that change the figure shows, for a **rename, move or delete**, who depends on it and **what would catch the failure**:
the compiler, the first request of a page, starting the application, a test on that path, a link check, or nothing. If the names stay and only the behaviour
changes, none of these failures happens, but callers may behave differently, and only tests tell.

In [ ]:
late = all_edges.assign(mode=all_edges["kind"].map(FAILURE_MODE)).query("mode in @LATE_MODES")
focus_candidates = late.groupby("target")["source"].nunique().rename("dependents the compiler does not check").nlargest(TOP_N).to_frame()
focus = FOCUS_FILES or focus_candidates.index[:1].tolist()
assert focus, "no focus file: the edge list has no dependents of the late kinds"
assert set(focus) <= tracked, f"focus files that are not tracked: {sorted(set(focus) - tracked)}"
focus_candidates

### Collect who depends on the focus files, by how they fail

The dependents are taken from all edges, the code graph included, and each gets a `mode` from its kind: the compiler (code graph), the first request (template edge), start-up, run
time, contract or name; the other users of the tables the focus files use are added as a schema change.

In [ ]:
from_edges = all_edges.query("target in @focus and layer == 'structure'").assign(mode=lambda d: d["kind"].map(FAILURE_MODE)).dropna(subset=["mode"])
focus_tables = all_edges.query("kind == 'data-table' and source in @focus")["target"].unique()
from_tables = (all_edges.query("kind == 'data-table' and target in @focus_tables and source not in @focus")
                        .assign(mode="schema", evidence=lambda d: "same table " + d["target"].str.removeprefix("table:")))
inbound = (pd.concat([from_edges, from_tables])[["source", "target", "kind", "mode", "evidence"]]
             .query("source not in @focus").drop_duplicates(["source", "mode"]))
assert set(inbound["mode"]) <= set(MODE_ORDER), set(inbound["mode"]) - set(MODE_ORDER)
by_mode = inbound.groupby("mode")["source"].nunique().reindex(MODE_ORDER, fill_value=0)
outbound = all_edges.query("source in @focus and layer == 'structure'").drop_duplicates(["target", "kind"])
by_mode.rename("files that depend on the focus").to_frame().assign(caught_by=lambda d: d.index.map(MODE_CAUGHT_BY))

**Figure: what could break.** One row per way of failing, from caught early (top) to caught late or never (bottom). The bar is the number of files that depend
on the focus in that way, the text names examples and what would catch it. Blue rows are caught before the code runs for a user, orange rows are not. The
last box lists what the focus files themselves depend on. The table above has the numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 0.85 * len(MODE_ORDER) + 2.2), facecolor=COLORS["surface"])
ax.set_facecolor(COLORS["surface"])
rows = len(MODE_ORDER)
ax.set_xlim(0, 13.5)
ax.set_ylim(-1.7, rows + 0.7)
ax.axis("off")
bar_x, bar_max = 5.0, 3.4
scale = bar_max / max(by_mode.max(), 1)
box_x, box_y = 10.6, rows / 2 - 0.4
ax.add_patch(FancyBboxPatch((box_x, box_y - 0.95), 2.7, 1.9, boxstyle="round,pad=0.03,rounding_size=0.1", fc="white", ec=COLORS["ink"], lw=1.2))
ax.text(box_x + 1.35, box_y + 0.55, "the change", ha="center", fontsize=10, color=COLORS["ink"])
ax.text(box_x + 1.35, box_y - 0.1, "\n".join(Path(f).name[:28] for f in focus[:3]), ha="center", va="center", fontsize=8.5, color=COLORS["ink2"])
for i, mode in enumerate(MODE_ORDER):
    y = rows - 1 - i
    n = int(by_mode[mode])
    color = COLORS["blue"] if mode in EARLY_MODES else COLORS["orange"]
    ax.text(0, y + 0.13, mode, fontsize=10.5, color=COLORS["ink"], va="center", fontweight="bold")
    ax.text(0, y - 0.17, "what catches it: " + MODE_CAUGHT_BY[mode], fontsize=8.5, color=COLORS["ink2"], va="center")
    ax.add_patch(FancyBboxPatch((bar_x, y - 0.18), max(n * scale, 0.05), 0.36, boxstyle="round,pad=0,rounding_size=0.04", fc=color if n else COLORS["gray"], ec="none"))
    ax.text(bar_x + max(n * scale, 0.05) + 0.12, y, f"{n}" if n else "none", va="center", fontsize=10 if n else 9, color=COLORS["ink"] if n else COLORS["ink2"])
    examples = inbound.query("mode == @mode")["source"].map(lambda p: Path(p).name).head(EXAMPLES_PER_MODE)
    if len(examples):
        ax.text(bar_x, y - 0.42, "e.g. " + ", ".join(examples), fontsize=8, color=COLORS["ink2"], va="center")
    if n:
        ax.add_patch(FancyArrowPatch((bar_x + bar_max + 0.9, y), (box_x - 0.05, box_y), arrowstyle="-|>", mutation_scale=9, lw=0.8, color=color, alpha=0.6))
out_text = "\n".join((outbound["kind"] + " -> " + outbound["target"].map(lambda p: Path(p).name)).head(5)) or "nothing found"
ax.text(box_x, box_y - 1.2, "the change itself depends on:\n" + out_text, fontsize=8, color=COLORS["ink2"], va="top")
ax.set_title("If the names change, these fail, and this is what catches them", loc="left", color=COLORS["ink"], fontsize=12, pad=26)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLORS["blue"]), plt.Rectangle((0, 0), 1, 1, color=COLORS["orange"])],
          labels=["caught before the code runs for a user", "caught late, only by a test, or not at all"], frameon=False, loc="lower left",
          bbox_to_anchor=(0, 1.0), ncol=2, fontsize=9, labelcolor=COLORS["ink2"])
if SAVE_FIGURES:
    fig.savefig(OUTPUT_PATH / "what-could-break.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
plt.show()

### The files behind the figure

The same dependents as a table, ready to read as a to-do list for the change: which file, how it depends, and the evidence line.

In [ ]:
inbound.assign(mode=pd.Categorical(inbound["mode"], MODE_ORDER, ordered=True)).sort_values(["mode", "source"])[["mode", "source", "kind", "evidence"]].head(60)

### The dependents in the code

Some of the dependents that the compiler does not check, shown as they look in the files: the line in the dependent that names the focus, and the line it names. One per
way of failing that has dependents, up to `EXAMPLES_PER_KIND` per way.

In [ ]:
shown = inbound.query("mode in @LATE_MODES").groupby("mode").head(EXAMPLES_PER_KIND)
if shown.empty:
    print("no dependents of the late kinds for this focus")
for r in shown.itertuples():
    lines = [int(x) for x in re.findall(r"line (\d+)", r.evidence)]
    show_example(f"{r.mode}: {r.kind}", r.source, lines[0] if lines else 0, focus[0], lines[1] if len(lines) > 1 else 1)

## Hidden dependencies: files that change together

**How it is found.** The Git history gives, for every pair of files, how often they changed in the same commit and in the same ticket (the ticket number is read from
the commit message). The **degree of coupling** is the shared changesets divided by the average number of changesets of the two files. A pair is then
checked against everything known: the code graph, the use edges, a URL, key or name, a table both use. A pair with none of these is a candidate for a
**hidden dependency**.

**The dependency problem.** Things that must change together and that nobody wrote down: a copy that evolved in parallel, a shared assumption, a contract
through a third file. Coupling is evidence, never proof: a release ticket, one person's habit or chance can explain it as well. For each pair the report therefore
lists **patterns**, possible reasons with their evidence.

**What catches it.** Nothing at build time. This is what the history alone can tell.

In [ ]:
coupling = pd.read_csv(COUPLING_CSV) if COUPLING_CSV else pd.DataFrame()
print(f"{len(coupling):,} coupled pairs of files read from {COUPLING_CSV}; the first five:")
coupling.head()

### Check: is the coupling file usable?

Skipped when no coupling file is configured. Otherwise the columns used below must exist and every pair must be named once, in a fixed order, so that a pair does
not appear twice.

In [ ]:
if len(coupling):
    needed = {"file_a", "file_b", "known_by", "hidden", "patterns", "degree_commits", "degree_tickets", "shared_commits", "shared_tickets"}
    assert needed <= set(coupling.columns), f"missing columns: {needed - set(coupling.columns)}"
    assert not coupling.duplicated(["file_a", "file_b"]).any(), coupling[coupling.duplicated(["file_a", "file_b"], keep=False)].head()
    assert (coupling["file_a"] < coupling["file_b"]).all(), "pairs must be in a fixed order"
    print(f"check passed: {len(coupling):,} pairs, columns present, no pair twice")

### How well are the coupled pairs explained?

Each pair is put into one of three classes by the strongest explanation found: **direct** (a code or use edge in one step, a URL, key or name, a table both
use), **indirect** (a path of two steps, a table reached through code, or only a shared dependency: weaker, because a hub such as one popular table links almost anything), or
**hidden** (none of these).

In [ ]:
direct_labels = ("code graph or use edge (1 step)", "shared table", "URL, key or shared name")
explained = (coupling.assign(strength=np.select([coupling["known_by"].isin(direct_labels), coupling["hidden"] == "yes"], ["direct", "hidden"], default="indirect"))
             if len(coupling) else coupling)
strength = explained["strength"].value_counts().reindex(["direct", "indirect", "hidden"], fill_value=0) if len(coupling) else pd.Series(dtype=int)
assert strength.sum() == len(explained), "a pair is missing from the classes"
strength.rename("pairs").to_frame()

**Figure: explained and hidden pairs.** One bar split into three parts, labelled directly, in the order direct, indirect, hidden. The hidden part is orange because
it is the one to look at; the table above has the counts.

In [ ]:
if len(coupling):
    fig, ax = plt.subplots(figsize=(10, 1.9), facecolor=COLORS["surface"])
    ax.set_facecolor(COLORS["surface"])
    shades = {"direct": COLORS["blue"], "indirect": COLORS["gray"], "hidden": COLORS["orange"]}
    left = 0
    for name, n in strength.items():
        ax.barh(0, n, left=left, color=shades[name], height=0.5, edgecolor=COLORS["surface"], linewidth=2)
        if n:
            ax.text(left + n / 2, 0, f"{name}\n{n}", ha="center", va="center", fontsize=9, color="white" if name != "indirect" else COLORS["ink"])
        left += n
    ax.set_xlim(0, left)
    ax.set_yticks([])
    ax.set_xlabel("Coupled pairs of files", color=COLORS["ink2"])
    ax.set_title("How well the coupled pairs are explained by known dependencies", loc="left", color=COLORS["ink"], fontsize=12)
    ax.tick_params(colors=COLORS["ink2"], labelsize=9)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    if SAVE_FIGURES:
        fig.savefig(OUTPUT_PATH / "explained-pairs.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
    plt.show()

### Why might they change together? The patterns

A pair can carry several patterns or none. Each one is a hypothesis to confirm or drop: a **clone** (copies that evolved in parallel), a **shared vocabulary** (rare
words in both), a **naming family** (variants of one thing), a **vertical slice** (different layers sharing a concept), a **burst** (one event), **persistent**
(recurs over a year or more), **one person** (one author made nearly all the shared changes), **bulk change** (large mechanical commits). The table counts the patterns among all
coupled pairs and among the hidden ones.

In [ ]:
if len(coupling):
    exploded = explained.assign(pattern=explained["patterns"].str.split(", ")).explode("pattern")
    pattern_counts = (exploded.groupby("pattern").agg(all_pairs=("file_a", "size"), hidden_pairs=("hidden", lambda s: int((s == "yes").sum())))
                              .sort_values("all_pairs", ascending=False))
    assert pattern_counts["all_pairs"].sum() == len(exploded)
    display(pattern_counts)

**Figure: patterns among all and among hidden pairs.** Two series, so a legend; the counts sit at the bar ends. Gray is every coupled pair, orange the hidden ones.

In [ ]:
if len(coupling):
    shown = pattern_counts.drop(index="none found", errors="ignore").sort_values("all_pairs")
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(shown) + 1.4), facecolor=COLORS["surface"])
    ax.set_facecolor(COLORS["surface"])
    y = np.arange(len(shown))
    bars_all = ax.barh(y + 0.19, shown["all_pairs"], height=0.34, color=COLORS["gray"], label="all coupled pairs")
    bars_hidden = ax.barh(y - 0.19, shown["hidden_pairs"], height=0.34, color=COLORS["orange"], label="hidden pairs")
    ax.bar_label(bars_all, padding=3, fontsize=8, color=COLORS["ink2"])
    ax.bar_label(bars_hidden, padding=3, fontsize=8, color=COLORS["ink2"])
    ax.set_yticks(y, shown.index)
    ax.set_xlabel("Pairs of files", color=COLORS["ink2"])
    ax.set_title("Patterns that may explain why files change together", loc="left", color=COLORS["ink"], fontsize=12)
    ax.tick_params(colors=COLORS["ink2"], labelsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=9, labelcolor=COLORS["ink2"], loc="lower right")
    if SAVE_FIGURES:
        fig.savefig(OUTPUT_PATH / "coupling-patterns.png", dpi=150, bbox_inches="tight", facecolor=COLORS["surface"])
    plt.show()

### The hidden pairs, strongest first

The pairs with no known dependency, ordered by the strength of the coupling, with their patterns. These are the first to ask a person about: each one names two
files that have to change together and that nothing connects.

In [ ]:
if len(coupling):
    hidden_pairs = (explained.query("hidden == 'yes'").assign(strength_sum=lambda d: d["degree_commits"] + d["degree_tickets"])
                             .sort_values("strength_sum", ascending=False)
                             [["file_a", "file_b", "shared_commits", "degree_commits", "shared_tickets", "degree_tickets", "patterns", "similarity", "shared_words"]])
    display(hidden_pairs.head(HIDDEN_ROWS))

### The hidden pairs in the code

The strongest hidden pairs shown in the files: the first rare word that both share, with its line in each file. This is the quickest way to see what the two files
have in common that no dependency records.

In [ ]:
if len(coupling):
    for r in hidden_pairs.dropna(subset=["shared_words"]).head(EXAMPLES_PER_KIND + 1).itertuples():
        word = r.shared_words.split()[0]
        show_example("changes together", r.file_a, find_line(r.file_a, re.escape(word)), r.file_b, find_line(r.file_b, re.escape(word)),
                     note=f"({r.shared_commits} commits, {r.shared_tickets} tickets) shared word `{word}`")

## Write the tables

The summary tables are written next to the figures so that they can be used without running the notebook. Nothing outside `output/` is written.

In [ ]:
per_kind.to_csv(OUTPUT_PATH / "edges-per-kind.csv")
type_matrix.to_csv(OUTPUT_PATH / "artifact-type-matrix.csv")
flows.to_csv(OUTPUT_PATH / "layer-flows.csv", index=False)
inbound.to_csv(OUTPUT_PATH / "what-could-break.csv", index=False)
if len(coupling):
    hidden_pairs.to_csv(OUTPUT_PATH / "hidden-pairs.csv", index=False)
print("written to", OUTPUT_PATH.resolve(), ":", sorted(p.name for p in OUTPUT_PATH.glob("*.csv")))

### Check: do the written tables match what was computed?

The per-kind counts are read back and compared with the edge list, and the figures must exist when they were requested.

In [ ]:
reread = pd.read_csv(OUTPUT_PATH / "edges-per-kind.csv", index_col="kind")
assert reread["edges"].sum() == len(all_edges), "the written per-kind table does not add up to the edge list"
if SAVE_FIGURES:
    missing_figures = [n for n in ("edges-per-kind.png", "artifact-type-matrix.png", "layer-dependencies.png", "what-could-break.png") if not (OUTPUT_PATH / n).exists()]
    assert not missing_figures, missing_figures
print(f"check passed: the per-kind table adds up to the {len(all_edges):,} edges" + (" and all figures are written" if SAVE_FIGURES else ""))

## Summary of the approach

The notebook reads the edge list and the coupled pairs that the `dependency-map` scripts wrote, checks that they match the repository, and describes each kind of dependency by
how it is searched, what breaks when its target changes and what would catch the failure. Three illustrations show what depends on what: the matrix of artifact types,
the layers with their arrows, and, for a chosen change, the files that depend on it sorted by how they fail. The history part asks which files change together although no known
dependency connects them, and lists the patterns that may explain it. Everything that is a heuristic is marked as one, and the spot-check tables are there for a person to
read against the source.

**Open and uncertain.** The edges of the heuristic kinds are leads, not proof; their precision was judged on small samples. Pairs below the coupling thresholds and
files with few revisions are not judged. Names that are assembled at run time are flagged, not resolved.